# Iteration 7: one Jetson-feasible decision, put together and checked on every set

Combines what survived Iterations 1-6 into a policy that fits the Jetson's 22 s budget, and checks each piece on every set (choosing: official,
NUDT 101; reporting: NUDT 202, the fresh NUDT 404).

Pieces:
1. **M per query.** The Jetson runs M = 5. Iteration 1 found that agreement at M = 5 made 2 wrong answers on NUDT 101 (fewer candidates per query,
   so two queries can agree on a false place when the right one dropped out). Options: M = 10 (slower per query, fewer queries fit), or M = 5 with a
   stricter agreement (each agreeing query needs >= 4 or 5 inliers).
2. **Agreement** among the 6 standard queries (free: no extra query).
3. **Extra queries** in the greedy order of Iteration 1 (chosen on official + NUDT 101), as many as the budget allows, answered at >= 11 inliers.
4. **Early exit**: answer as soon as the pooled choice has >= 30 inliers (saves time, the budget is per photo).
5. **Late verifier** (Iteration 2, logistic, trained on official + NUDT 101): only for photos still skipped, on the 6-query choice.
6. **Attitude correction** (Iteration 3): official camera only, fitted held out by flight.

**Cost model (assumption, no Jetson here):** the submitted run took 13.06 s mean at M = 5, mostly 6 queries -> 2.2 s per query. The share of the
fine stage on the Jetson is unknown (the coarse FFT runs as a matrix multiply there because cuFFT fails, so it dominates); M = 10 is shown at
2.6 s and 3.0 s per query. Cap = floor(22 s / seconds per query).

In [1]:
import os, sys
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)
from train.checks.replay import load_set, truncate_m, score, paired_bootstrap, points, error_sigma, STANDARD
from train.checks.query_policy import run_policy
from train.checks.verifier import oof_probability, pick_threshold, fit_predict, feature_matrix
from train.checks.pose_bias import fit as fit_bias, apply as apply_bias
from sklearn.model_selection import GroupKFold
DUMPS = ROOT / "tests/Test_Iteration_6/results/rich_dump"; OUT = ROOT / "tests/Test_Iteration_6/results"
SET_NAMES = {"official": "official", "nudt101": "nudt_seed101", "nudt202": "nudt_seed202", "nudt404": "nudt_fresh404"}
CHOOSE_ON, REPORT_ON = ["official", "nudt101"], ["nudt202", "nudt404"]
sets = {n: load_set(DUMPS / f) for n, f in SET_NAMES.items() if (DUMPS / f).exists()}
greedy = pd.read_csv(OUT / "it1_greedy_order.csv")
ORDER = [(float(h), float(s)) for h, s in zip(greedy.heading, greedy.scale)]
SECOND_PASS = [(h, s) for s in (1.0, 1.05) for h in (-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0)]
print({k: len(v) for k, v in sets.items()}, "greedy order:", ORDER[:6])

{'official': 379, 'nudt101': 300, 'nudt202': 200, 'nudt404': 200} greedy order: [(-7.5, 0.95), (-2.5, 1.0), (0.0, 0.95), (2.5, 0.95), (7.5, 0.95), (-10.0, 0.95)]


## 1. Verifier probabilities (late, on the 6-query choice) and the attitude model

In [2]:
rows = {n: pd.read_csv(OUT / ("it2_rows_%s.csv" % n)) for n in sets if (OUT / ("it2_rows_%s.csv" % n)).exists()}
choose_rows = pd.concat([rows[n] for n in CHOOSE_ON], ignore_index=True)
p_oof = oof_probability(choose_rows, "logistic"); T_STD = pick_threshold(choose_rows, p_oof, "std6")
p_by = {}
for n, r in rows.items():
    if n in CHOOSE_ON:
        sub = choose_rows.assign(p=p_oof); sub = sub[(sub.data == n) & (sub.step == "std6")]
    else:
        p, _ = fit_predict(choose_rows, r, "logistic"); sub = r.assign(p=p); sub = sub[sub.step == "std6"]
    p_by[n] = dict(zip(sub.photo, sub.p))
print("verifier cut on the 6-query choice: P(wrong) < %.3f" % T_STD)

verifier cut on the 6-query choice: P(wrong) < 0.256


In [3]:
def evaluate(name, M, agree_q, cap, early, verifier, attitude):
    photos = sets[name]
    rows_out = []
    for pid, ph in photos.items():
        p5 = truncate_m(ph, M) if M < 10 else ph
        r = run_policy(p5, wide_order=ORDER, max_queries=cap, early_bar=early or None, agree_query_inliers=agree_q)
        if not r["answered"] and verifier and M == 10:
            pv = p_by.get(name, {}).get(pid, np.nan)
            if np.isfinite(pv) and pv < T_STD:
                from train.checks.query_policy import _choice
                ids = [i for i, q in enumerate(ph["queries"]) if (q["heading"], q["scale"]) in STANDARD]
                x, y, inl = _choice(ph, ids)
                if x is not None:
                    err = error_sigma(ph, x, y); r.update(answered=True, points=points(err), err_sigma=err, stage="late_verifier", x=x, y=y)
        r.update(photo=pid, ex=(r["x"] - ph["truth"][0]) if r["x"] is not None else 0.0, ey=(r["y"] - ph["truth"][1]) if r["y"] is not None else 0.0,
                 sigma=ph["sigma"], h=ph["height"], yaw=ph["yaw"], pitch=ph["pitch"], roll=ph["roll"])
        rows_out.append(r)
    t = pd.DataFrame(rows_out); t["flight"] = t.h.round(0)
    if attitude and name == "official":
        newp = t.points.to_numpy().copy()
        for tr, te in GroupKFold(n_splits=5).split(t, groups=t.flight):
            train = t.iloc[tr]; train = train[train.answered].rename(columns={"err_sigma": "err_sigma"})
            m = fit_bias(train.assign(err_sigma=train.err_sigma), ("angle", "const"))
            test = t.iloc[te]; ex, ey = apply_bias(test, m)
            err = np.hypot(ex, ey) / test.sigma.to_numpy()
            newp[te] = [points(e) if a else 0 for e, a in zip(err, test.answered)]
        t["points"] = newp
    return t
def jetson_today(name):
    return pd.DataFrame([dict(photo=pid, **run_policy(truncate_m(ph, 5), agree_min=99, escalation_bar=8, wide_order=SECOND_PASS, max_queries=10))
                         for pid, ph in sets[name].items()])

## 2. The grid (every setting on every set)

`jetson s` = mean queries x seconds per query; `over 22 s` = photos whose queries would not fit (0 by construction when the cap holds).

In [4]:
configs = []
for M, s_q in ((5, 2.2), (10, 2.6), (10, 3.0)):
    cap = int(22.0 // s_q)
    for agree_q in ((3, 4, 5) if M == 5 else (3,)):
        for early in (0, 30):                       # 0 = no early exit (pivot tables drop None)
            for verifier in ((False, True) if M == 10 else (False,)):
                for attitude in (False, True):
                    configs.append(dict(M=M, s_per_query=s_q, cap=cap, agree_q=agree_q, early=early, verifier=verifier, attitude=attitude))
base = {n: jetson_today(n) for n in sets}
results = []
for cfg in configs:
    for n in sets:
        t = evaluate(n, cfg["M"], cfg["agree_q"], cfg["cap"], cfg["early"], cfg["verifier"], cfg["attitude"])
        g = paired_bootstrap(base[n].points.to_numpy(), t.points.to_numpy())
        results.append(dict(cfg, set=n, score=round(score(t.points), 2), wrong=int((t.points < 0).sum()), answered=round(t.answered.mean(), 3),
                            mean_q=round(t.n_queries.mean(), 2), jetson_s=round(t.n_queries.mean() * cfg["s_per_query"], 1),
                            jetson_today=round(score(base[n].points), 2), gain_vs_jetson_today="%+.2f [%+.2f, %+.2f]" % g))
res = pd.DataFrame(results)
res.to_csv(OUT / "it7_jetson_policies.csv", index=False)
show = res.pivot_table(index=["M", "s_per_query", "cap", "agree_q", "early", "verifier", "attitude"], columns="set", values=["score", "wrong"], aggfunc="first")
display(show)

score                            wrong                         
set                                                nudt101 nudt202 nudt404 official nudt101 nudt202 nudt404 official
M  s_per_query cap agree_q early verifier attitude                                                                  
5  2.2         9   3       0     False    False      54.44   55.98   55.53    53.38       2       0       1        0
                                          True       54.44   55.98   55.53    54.27       2       0       1        0
                           30    False    False      54.36   55.86   55.53    53.19       2       0       1        0
                                          True       54.36   55.86   55.53    54.27       2       0       1        0
                   4       0     False    False      55.36   55.98   55.53    52.12       0       0       1        0
                                          True       55.36   55.98   55.53    52.94       0       0       1        0
                           30    False    False      55.28   55.86   55.53    51.93       0       0       1        0
                                          True       55.28   55.86   55.53    53.00       0       0       1        0
                   5       0     False    False      54.96   55.98   55.53    50.75       0       0       1        0
                                          True       54.96   55.98   55.53    51.51       0       0       1        0
                           30    False    False      54.88   55.86   55.53    50.56       0       0       1        0
                                          True       54.88   55.86   55.53    51.58       0       0       1        0
10 2.6         8   3       0     False    False      55.16   56.28   53.43    53.22       0       0       4        0
                                          True       55.16   56.28   53.43    54.11       0       0       4        0
                                 True     False      55.16   56.28   53.61    55.73       0       0       4        0
                                          True       55.16   56.28   53.61    56.61       0       0       4        0
                           30    False    False      55.08   56.16   53.43    53.03       0       0       4        0
                                          True       55.08   56.16   53.43    54.11       0       0       4        0
                                 True     False      55.08   56.16   53.61    55.54       0       0       4        0
                                          True       55.08   56.16   53.61    56.61       0       0       4        0
   3.0         7   3       0     False    False      54.64   56.28   53.13    53.22       0       0       4        0
                                          True       54.64   56.28   53.13    54.11       0       0       4        0
                                 True     False      54.64   56.28   53.31    55.73       0       0       4        0
                                          True       54.64   56.28   53.31    56.61       0       0       4        0
                           30    False    False      54.56   56.16   53.13    53.03       0       0       4        0
                                          True       54.56   56.16   53.13    54.11       0       0       4        0
                                 True     False      54.56   56.16   53.31    55.54       0       0       4        0
                                          True       54.56   56.16   53.31    56.61       0       0       4        0

## 3. Settings with 0 wrong on every set, best first (by the choosing sets), with the gain on the report sets

In [5]:
key = ["M", "s_per_query", "cap", "agree_q", "early", "verifier", "attitude"]
ok = res.groupby(key).filter(lambda g: (g.wrong == 0).all())
rank = ok[ok.set.isin(CHOOSE_ON)].groupby(key).score.mean().sort_values(ascending=False)
top = rank.head(8).reset_index()[key]
display(top.merge(res, on=key).pivot_table(index=key, columns="set", values=["score", "gain_vs_jetson_today", "jetson_s"], aggfunc="first"))

M,s_per_query,cap,agree_q,early,verifier,attitude
